In [1]:
%%time

import tensorflow as tf
print(tf.__version__)
import sys
sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np


def plot_loss(history, *losses):
    for loss in losses:
        plt.plot(history.history[loss], label=loss)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def scaling(x, min, max):
    return np.where(x < min, 0.0, np.where(x > max, 1.0, (x - min) / (max - min)))

early_stopping = EarlyStopping(
    monitor='val_loss',  # 
    patience=50,        # 
    verbose=1,          # 
    mode='min',         # 
    restore_best_weights=True  # 
)
from keras.callbacks import Callback

2024-01-26 00:57:20.318907: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-01-26 00:57:20.384866: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-01-26 00:57:20.384904: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-01-26 00:57:20.384951: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-01-26 00:57:20.397357: I tensorflow/core/platform/cpu_feature_g

2.14.0
CPU times: user 3.22 s, sys: 2.14 s, total: 5.37 s
Wall time: 2.97 s


In [2]:
SAVE_DIR = "../data"
file_criteo = SAVE_DIR + "/criteo-uplift-v2.1.csv"
df_criteo_ori = pd.read_csv(file_criteo, sep=',')

In [4]:
%%time
model_name = "CF"
result_list = []
DRP_aucc_test_list = []
roi_rank_pre_test_list = []


for sample in np.arange(0.11, 0.22, 0.01):
    print("sample_value = ", sample)
    random_state=20220720
    df_criteo=df_criteo_ori.sample(frac=sample, random_state=random_state).reset_index(drop=True)
    X = df_criteo[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']].values

    X[:, 0] = scaling(X[:, 0], min=np.min(X[:, 0]), max=np.max(X[:, 0]))
    X[:, 1] = scaling(X[:, 1], min=np.min(X[:, 1]), max=np.max(X[:, 1]))
    X[:, 2] = scaling(X[:, 2], min=np.min(X[:, 2]), max=np.max(X[:, 2]))
    X[:, 3] = scaling(X[:, 3], min=np.min(X[:, 3]), max=np.max(X[:, 3]))
    X[:, 4] = scaling(X[:, 4], min=np.min(X[:, 4]), max=np.max(X[:, 4]))
    X[:, 5] = scaling(X[:, 5], min=np.min(X[:, 5]), max=np.max(X[:, 5]))
    X[:, 6] = scaling(X[:, 6], min=np.min(X[:, 6]), max=np.max(X[:, 6]))
    X[:, 7] = scaling(X[:, 7], min=np.min(X[:, 7]), max=np.max(X[:, 7]))
    X[:, 8] = scaling(X[:, 8], min=np.min(X[:, 8]), max=np.max(X[:, 8]))
    X[:, 9] = scaling(X[:, 9], min=np.min(X[:, 9]), max=np.max(X[:, 9]))
    X[:, 10] = scaling(X[:, 10], min=np.min(X[:, 10]), max=np.max(X[:, 10]))
    X[:, 11] = scaling(X[:, 11], min=np.min(X[:, 11]), max=np.max(X[:, 11]))

    T = df_criteo['treatment'].values.reshape(-1, 1)
    Y_visit = df_criteo['visit'].values.reshape(-1, 1)
    Y_conv = df_criteo['conversion'].values.reshape(-1, 1)

    T.shape, Y_visit.shape, Y_conv.shape

    # calculate len
    train_len = int(len(X) * 0.70)
    cali_len = int(len(X) * 0.0)
    test_len = len(X) - train_len - cali_len

    # obtain train set
    X_train = X[:train_len, :]
    T_train = T[:train_len, :]
    Y_visit_train = Y_visit[:train_len, :]
    Y_conv_train = Y_conv[:train_len, :]

    # obtain calibration set
    X_cali = X[train_len:train_len+cali_len, :]
    T_cali = T[train_len:train_len+cali_len, :]
    Y_visit_cali = Y_visit[train_len:train_len+cali_len, :]
    Y_conv_cali = Y_conv[train_len:train_len+cali_len, :]

    # obtain test set
    X_test = X[train_len+cali_len:, :]
    T_test = T[train_len+cali_len:, :]
    Y_visit_test = Y_visit[train_len+cali_len:, :]
    Y_conv_test = Y_conv[train_len+cali_len:, :]

    #print(train_len, X_train.shape, X_test.shape, len(X), X_cali.shape, T_train.shape, Y_visit_train.shape)
    
    
    sys.path.append("..")
    from model.uplift_model import *

    
    import tensorflow as tf
    from keras.callbacks import LearningRateScheduler
    from keras.callbacks import ReduceLROnPlateau
    from model.roi_model import *
    
    
    # GRF
    from econml.dml import CausalForestDML
    from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
    from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
    from sklearn.dummy import DummyRegressor, DummyClassifier
    import pickle

    import sklearn 
    import sklearn.metrics
    from metric.Metric import *
    import keras
    import keras.backend as K
    import tensorflow as tf
    from keras.callbacks import LearningRateScheduler
    from keras.callbacks import ReduceLROnPlateau
    from model.uplift_model import *

    grf_aucc_list = []

    X_grf_train = X_train
    T_grf_train = T_train.flatten()
    Y_visit_grf_train = Y_visit_train.flatten()
    Y_conv_grf_train = Y_conv_train.flatten()

    X_grf_test = X_test
    T_grf_test = T_test.flatten()
    Y_visit_grf_test = Y_visit_test.flatten()
    Y_conv_grf_test = Y_conv_test.flatten()

    
    depth = 6
    min_samples_leaf = 100
    n_estimators = 128

    grf_random_state = 20220723
    
    # visit
    est_it = CausalForestDML(model_y=RandomForestClassifier(n_estimators=80, max_samples = 0.7, random_state=grf_random_state, max_depth=6, min_samples_leaf=500, n_jobs=32),
                          model_t=RandomForestClassifier(n_estimators=80, max_samples = 0.7, random_state=grf_random_state, max_depth=6, min_samples_leaf=500, n_jobs=32),
                          discrete_treatment=True,
                          cv=3,
                          n_estimators=n_estimators,
                          n_jobs=32,
                          max_depth=depth,
                          min_samples_leaf = min_samples_leaf,
                          random_state=grf_random_state)

    est_it.fit(Y_visit_grf_train, T_grf_train, X=X_grf_train, cache_values=True)
    
    model_file = "../model_file/uplift/criteo/final_model/grf/C_visit_CausalForestDML_{}.model".format(sample)
    
    fw = open(model_file, "wb")

    pickle.dump(est_it, fw)

    fw.close()
    
    grf_test_pre_visit = est_it.effect(X_grf_test)
    
    # conv
    est_it = CausalForestDML(model_y=RandomForestClassifier(n_estimators=160, max_samples = 0.7, random_state=grf_random_state, max_depth=12, min_samples_leaf=500, n_jobs=32),
                          model_t=RandomForestClassifier(n_estimators=160, max_samples = 0.7, random_state=grf_random_state, max_depth=12, min_samples_leaf=500, n_jobs=32),
                          discrete_treatment=True,
                          cv=3,
                          n_estimators=n_estimators,
                          n_jobs=32,
                          max_depth=depth,
                          min_samples_leaf = min_samples_leaf,
                          random_state=grf_random_state)

    est_it.fit(Y_conv_grf_train, T_grf_train, X=X_grf_train, cache_values=True)
    
    model_file = "../model_file/uplift/criteo/final_model/grf/C_conv_CausalForestDML_{}.model".format(sample)
    
    fw = open(model_file, "wb")

    pickle.dump(est_it, fw)

    fw.close()
    
    grf_test_pre_conv = est_it.effect(X_grf_test)
    
    # roi
    
    roi_rank_pre_test = grf_test_pre_conv / np.where(abs(grf_test_pre_visit) < 1e-6, 1e-6, grf_test_pre_visit)

    roi_rank_pre_test_list.append(roi_rank_pre_test)
    DRP_aucc = get_uplift_model_aucc_no_show(t=(T_test > 0.5).flatten(), y_reward=Y_conv_test.flatten(), y_cost=Y_visit_test.flatten(), roi_pred=roi_rank_pre_test.flatten(), quantile=200)
    DRP_aucc_test_list.append(DRP_aucc)
    result_list.append(DRP_aucc[0])
    
    

print(result_list)
print(np.mean(result_list))
print(np.std(result_list))

# store test aucc for pic 
import pandas as pd

def get_aucc_cost_curve(aucc_list):
    delta_cost_list_group = np.array([aucc[1] for aucc in aucc_list])
    delta_reward_list_group = np.array([aucc[2] for aucc in aucc_list])
    
    avg_delta_cost_list = np.mean(delta_cost_list_group, axis=0)
    avg_delta_reward_list = np.mean(delta_reward_list_group, axis=0)
    
    df_aucc_cost_curve = pd.DataFrame(avg_delta_cost_list, columns=['delta_cost'])
    df_aucc_cost_curve['delta_reward'] = avg_delta_reward_list
    
    return df_aucc_cost_curve

average_list = get_aucc_cost_curve(DRP_aucc_test_list)
print("avg-aucc = ", np.sum(average_list['delta_reward'].values) / (average_list['delta_reward'].values[-1] * 201))
average_list.to_csv("../figure/sigir/{}.csv".format(model_name))

sample_value =  0.11
AUCC =  0.6846866788645563
sample_value =  0.12
AUCC =  0.6716049443759213
sample_value =  0.13
AUCC =  0.6625387779777097
sample_value =  0.13999999999999999
AUCC =  0.6488392285168901
sample_value =  0.14999999999999997
AUCC =  0.5923515183828985
sample_value =  0.15999999999999998
AUCC =  0.583183764525853
sample_value =  0.16999999999999998
AUCC =  0.5693786006407239
sample_value =  0.17999999999999997
AUCC =  0.6404996485330231
sample_value =  0.18999999999999995
AUCC =  0.6423778170255983
sample_value =  0.19999999999999996
AUCC =  0.6225693926450866
sample_value =  0.20999999999999996
AUCC =  0.6580788902199669
[0.6846866788645563, 0.6716049443759213, 0.6625387779777097, 0.6488392285168901, 0.5923515183828985, 0.583183764525853, 0.5693786006407239, 0.6404996485330231, 0.6423778170255983, 0.6225693926450866, 0.6580788902199669]
0.6341917510643843
0.03609737765080648
avg-aucc =  0.634756033419906
CPU times: user 14h 25min 29s, sys: 33min 4s, total: 14h 58min 3

In [8]:
formatted_numbers = ["{:.4f}".format(number) for number in result_list]
output = " & ".join(formatted_numbers)

print(output)

0.6847 & 0.6716 & 0.6625 & 0.6488 & 0.5924 & 0.5832 & 0.5694 & 0.6405 & 0.6424 & 0.6226 & 0.6581
